# RQ4: a first look at policy scenarios

This is only a preview. It ties the adoption gaps from RQ1 to a simple what-if, to show the kind of question the final scenario work will answer properly.

## Setup (local or Google Colab)

On Colab this pulls the project data and code from GitHub. Locally it uses the repo folder you already have.

In [1]:
# Works whether you run this locally or on Google Colab.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.isdir('data'):
    import subprocess
    subprocess.run(['git','clone','-q','https://github.com/bcudjoe/mobile-money-ghana-forecasting.git'])
    os.chdir('mobile-money-ghana-forecasting')
print('Colab' if IN_COLAB else 'local')

local


In [2]:
import pandas as pd, numpy as np
fx=pd.read_csv('data/processed/findex_ghana_clean.csv')
base=fx['adopts_mm'].mean()*100
print(f'Baseline adoption (pooled): {base:.1f}%')

Baseline adoption (pooled): 74.8%


## Where the gaps are, by income and education

In [3]:
by_income=(fx.groupby('income_quintile').adopts_mm.mean()*100).round(1)
by_edu=(fx.groupby('education').adopts_mm.mean()*100).round(1)
print('by income quintile:\n', by_income.to_string())
print('\nby education:\n', by_edu.to_string())

by income quintile:
 income_quintile
1    56.3
2    66.5
3    73.1
4    79.6
5    88.1

by education:
 education
1.0    55.1
2.0    80.4
3.0    96.7


## Scenario: close half the gap for the two poorest income groups

If targeted agent expansion and better connectivity lifted adoption in quintiles 1 and 2 halfway to the top group's rate, what would overall adoption look like? The flip below is illustrative, not a model estimate.

In [4]:
top=by_income.max()
sim=fx.copy()
for q in [1,2]:
    mask=sim['income_quintile']==q
    cur=sim.loc[mask,'adopts_mm'].mean()*100
    target=cur+(top-cur)/2
    # randomly flip enough non-adopters to hit the target rate (illustrative)
    n_need=int(round((target-cur)/100*mask.sum()))
    non=sim[mask & (sim.adopts_mm==0)].index
    flip=np.random.default_rng(1).choice(non, size=min(n_need,len(non)), replace=False)
    sim.loc[flip,'adopts_mm']=1
new=sim['adopts_mm'].mean()*100
print(f'Scenario adoption: {new:.1f}%  (baseline {base:.1f}%, gain {new-base:+.1f} pp)')

Scenario adoption: 79.1%  (baseline 74.8%, gain +4.3 pp)


For the final report this rough flip is replaced with estimates from the fitted driver model and the forecasts, plus a regional view of inclusion.